
Question: Retrieve the top 2 most recent orders for each customer. 


In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number

# Create a SparkSession
spark = SparkSession.builder.appName("Top3RecentOrders").getOrCreate()

# Sample data
data = [
    (1, 101, "2023-01-15"),
    (1, 101, "2023-07-15"),
    (1, 101, "2023-07-16"),
    (2, 102, "2023-02-20"),
    (1, 103, "2023-03-10"),
    (3, 104, "2023-04-05"),
    (2, 105, "2023-05-12"),
    (2, 105, "2023-05-12")
]

# Define the schema
schema = ["CUSTOMERID", "ORDERID", "ORDERDATE"]

# Create a DataFrame
df = spark.createDataFrame(data, schema)

# Convert the ORDERDATE column to a date type
df = df.withColumn("ORDERDATE", df["ORDERDATE"].cast("date"))

# Define a window specification partitioned by CUSTOMERID and ordered by ORDERDATE in descending order
window_spec = Window.partitionBy("CUSTOMERID").orderBy(df["ORDERDATE"].desc())

# Add a rank column based on ORDERDATE within each customer's partition
df = df.withColumn("rank", row_number().over(window_spec))

# Filter the results to include only the top 3 most recent orders for each customer
top3_recent_orders = df.filter(df["rank"] <= 3)

# Show the result
top3_recent_orders.show()


+----------+-------+----------+----+
|CUSTOMERID|ORDERID| ORDERDATE|rank|
+----------+-------+----------+----+
|         1|    101|2023-07-16|   1|
|         1|    101|2023-07-15|   2|
|         1|    103|2023-03-10|   3|
|         2|    105|2023-05-12|   1|
|         2|    105|2023-05-12|   2|
|         2|    102|2023-02-20|   3|
|         3|    104|2023-04-05|   1|
+----------+-------+----------+----+

